# Satellite Image Segmentation Using Vision Transformers

Fine-tune **SegFormer** (a hierarchical vision transformer) for pixel-wise segmentation of satellite or drone imagery. Classes include background, buildings, roads, water, barren land, forest, agricultural land, fire, smoke, other, occluded/unknown, industrial/factory and floodwater. This notebook covers data preparation, paired transforms, training, validation metrics, and visual inspection of predictions.

> **Data required:** image tiles and corresponding single-channel class-ID masks. Organize them in the folder structure below. A mask pixel must contain its class index (`0` to `NUM_CLASSES - 1`); use `IGNORE_INDEX` (255) for unlabeled pixels.

```text
data/
  train/images/tile_001.png
  train/masks/tile_001.png
  val/images/tile_101.png
  val/masks/tile_101.png
```

Images and masks are paired by filename stem. Images can be RGB PNG/JPEG; masks should be PNG and encode class IDs 0–12 in the order defined in the configuration cell. Update the paths there for your dataset.

**Fire and smoke require labeled examples:** the model will not learn to identify them from land-cover-only masks. Annotate fire pixels as class 7 and smoke pixels as class 8 across varied altitudes, lighting, seasons, and sensor conditions. Class 9 (`other`) is for visible categories not represented by the other labels; class 0 is background; class 10 (`occluded/unknown`) marks pixels where smoke, fog, cloud, or another obstruction hides the ground; class 11 (`industrial/factory`) marks factories and industrial sites; class 12 (`flood`) marks floodwater. Label ordinary water as class 3 and floodwater as class 12 so the model can learn the distinction. The model must be trained with masks using these IDs. This notebook uses RGB input, so it cannot use thermal heat signatures. High-altitude imagery may also make small fires or thin smoke plumes too small to resolve.


## 1. Install dependencies

Run once in a fresh Colab or notebook environment. On a local machine, install these packages in the active environment.

In [ ]:
%pip install -q torch torchvision transformers accelerate albumentations opencv-python-headless scikit-learn matplotlib pillow tqdm


## 2. Imports and configuration

In [ ]:
from pathlib import Path
import random
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerForSemanticSegmentation
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Change this to your dataset root. In Colab, mount Drive or upload data first.
DATA_ROOT = Path("data")
TRAIN_IMAGE_DIR = DATA_ROOT / "train" / "images"
TRAIN_MASK_DIR = DATA_ROOT / "train" / "masks"
VAL_IMAGE_DIR = DATA_ROOT / "val" / "images"
VAL_MASK_DIR = DATA_ROOT / "val" / "masks"

# Keep this order synchronized with the integer IDs encoded in every mask.
CLASS_NAMES = ["background", "building", "road", "water", "barren", "forest", "agricultural", "fire", "smoke", "other", "occluded/unknown", "industrial/factory", "flood"]
NUM_CLASSES = len(CLASS_NAMES)
IGNORE_INDEX = 255
MODEL_CHECKPOINT = "nvidia/segformer-b0-finetuned-ade-512-512"  # pretrained SegFormer; replace head for this dataset
IMAGE_SIZE = 512
BATCH_SIZE = 4
EPOCHS = 10
LEARNING_RATE = 6e-5
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Classes:", dict(enumerate(CLASS_NAMES)))


## 3. Check the dataset

The file stems in each image directory must match the stems in its mask directory. If your masks use RGB colors to encode classes, convert them to integer class IDs before training.

In [ ]:
from pathlib import Path

# Set this to the directory containing train/ and val/.
# Example in Colab: DATA_ROOT = Path("/content/drive/MyDrive/my_satellite_dataset")
DATA_ROOT = Path("data")
TRAIN_IMAGE_DIR = DATA_ROOT / "train" / "images"
TRAIN_MASK_DIR = DATA_ROOT / "train" / "masks"
VAL_IMAGE_DIR = DATA_ROOT / "val" / "images"
VAL_MASK_DIR = DATA_ROOT / "val" / "masks"


def list_pairs(image_dir, mask_dir):
    image_dir, mask_dir = Path(image_dir), Path(mask_dir)
    if not image_dir.exists() or not mask_dir.exists():
        raise FileNotFoundError(
            f"Expected directories: {image_dir} and {mask_dir}. "
            "Check DATA_ROOT above and ensure your dataset has train/ and val/ folders."
        )
    image_paths = sorted(
        p for p in image_dir.iterdir()
        if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
    )
    mask_by_stem = {
        p.stem: p for p in mask_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".png"
    }
    pairs = [(p, mask_by_stem[p.stem]) for p in image_paths if p.stem in mask_by_stem]
    missing = [p.name for p in image_paths if p.stem not in mask_by_stem]
    if missing:
        print(f"Warning: {len(missing)} images have no matching PNG mask; first few: {missing[:5]}")
    if not pairs:
        raise ValueError(
            f"No paired image/mask files found in {image_dir} and {mask_dir}. "
            "Image and mask filenames must share the same stem."
        )
    return pairs


train_pairs = list_pairs(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR)
val_pairs = list_pairs(VAL_IMAGE_DIR, VAL_MASK_DIR)
print(f"Training pairs: {len(train_pairs)} | validation pairs: {len(val_pairs)}")


## 4. Visualize a sample

In [ ]:
img_path, mask_path = train_pairs[0]
image = np.array(Image.open(img_path).convert("RGB"))
mask = np.array(Image.open(mask_path))
print("Image:", image.shape, image.dtype, "| mask:", mask.shape, mask.dtype)
print("Mask values (sample):", np.unique(mask)[:30])
print("Expected IDs:", {i: name for i, name in enumerate(CLASS_NAMES)}, "and ignore:", IGNORE_INDEX)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.imshow(image); plt.title("Satellite image"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(mask, cmap="tab20", vmin=0, vmax=NUM_CLASSES-1); plt.title("Class-ID mask"); plt.axis("off"); plt.show()


## 5. Dataset and paired transforms

Geometric changes are applied jointly to the image and mask so pixel labels remain aligned. Validation uses resize only. Masks are resized with nearest-neighbor interpolation by Albumentations.

In [ ]:
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class SatelliteSegmentationDataset(Dataset):
    def __init__(self, pairs, transform):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]
        image = np.array(Image.open(image_path).convert("RGB"))
        mask = np.array(Image.open(mask_path))
        if mask.ndim == 3:
            raise ValueError(f"Mask {mask_path} is RGB; convert colors to class IDs first.")
        mask = mask.astype(np.int64)
        invalid = np.setdiff1d(np.unique(mask), np.r_[np.arange(NUM_CLASSES), IGNORE_INDEX])
        if invalid.size:
            raise ValueError(f"Unexpected mask labels {invalid[:20]} in {mask_path}; expected 0..{NUM_CLASSES-1} or {IGNORE_INDEX}.")
        augmented = self.transform(image=image, mask=mask)
        return {"pixel_values": augmented["image"], "labels": augmented["mask"].long()}

train_ds = SatelliteSegmentationDataset(train_pairs, train_transform)
val_ds = SatelliteSegmentationDataset(val_pairs, val_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
print("Train batches:", len(train_loader), "| validation batches:", len(val_loader))


## 6. Initialize SegFormer

The MiT-B0 encoder starts from ImageNet-pretrained weights. The segmentation head is initialized for this dataset's class count. The first run downloads the checkpoint from Hugging Face.

In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
    id2label={i: name for i, name in enumerate(CLASS_NAMES)},
    label2id={name: i for i, name in enumerate(CLASS_NAMES)},
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


## 7. Training and validation

We report mean intersection-over-union (mIoU) and pixel accuracy over non-ignored pixels. For a larger project, consider class-weighted loss or sampling strategies if the classes are strongly imbalanced.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    confusion = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64)
    for batch in loader:
        pixels = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(pixel_values=pixels).logits
        logits = torch.nn.functional.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
        preds = logits.argmax(dim=1)
        valid = labels != IGNORE_INDEX
        indices = labels[valid] * NUM_CLASSES + preds[valid]
        confusion += torch.bincount(indices.cpu(), minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    intersection = confusion.diag().float()
    union = confusion.sum(0).float() + confusion.sum(1).float() - intersection
    iou = intersection / union.clamp_min(1)
    accuracy = intersection.sum() / confusion.sum().clamp_min(1)
    return iou.numpy(), float(accuracy), confusion

history = {"train_loss": [], "val_miou": [], "val_accuracy": []}
best_miou = -1.0
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in progress:
        pixels = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(pixel_values=pixels, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        running_loss += outputs.loss.item()
        progress.set_postfix(loss=f"{outputs.loss.item():.4f}")

    train_loss = running_loss / max(1, len(train_loader))
    iou, accuracy, confusion = evaluate(model, val_loader)
    mean_iou = float(np.nanmean(iou))
    history["train_loss"].append(train_loss)
    history["val_miou"].append(mean_iou)
    history["val_accuracy"].append(accuracy)
    print(f"Epoch {epoch+1}: train loss={train_loss:.4f}, val mIoU={mean_iou:.4f}, pixel acc={accuracy:.4f}")
    print("Per-class IoU:", {name: round(float(score), 4) for name, score in zip(CLASS_NAMES, iou)})

    if mean_iou > best_miou:
        best_miou = mean_iou
        model.save_pretrained(OUTPUT_DIR / "best_segformer")
        print("Saved best checkpoint to", OUTPUT_DIR / "best_segformer")


## 8. Plot learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], marker="o"); axes[0].set_title("Training loss"); axes[0].set_xlabel("Epoch")
axes[1].plot(history["val_miou"], marker="o", label="mIoU")
axes[1].plot(history["val_accuracy"], marker="o", label="Pixel accuracy")
axes[1].set_title("Validation metrics"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()


## 9. Inspect predictions

The model output has a lower spatial resolution than the input, so logits are resized to the mask dimensions before taking the class argmax.

In [ ]:
# Colors follow the supplied expected-output legend.
CLASS_COLORS = np.array([
    [245, 245, 245],  # background: light gray
    [255, 82, 82],    # building: red
    [255, 190, 0],    # road: yellow-orange
    [33, 150, 243],   # water: blue
    [161, 136, 127],  # barren: taupe
    [76, 175, 80],    # forest: green
    [139, 195, 74],   # agricultural: lime green
    [255, 30, 0],    # fire: bright red-orange
    [180, 180, 180], # smoke: gray
    [156, 39, 176],  # other: purple
    [55, 55, 55],    # occluded/unknown: dark gray
    [96, 125, 139],  # industrial/factory: slate blue-gray
    [121, 85, 52],   # flood: muddy brown
], dtype=np.uint8)


def colorize_mask(mask):
    mask = np.asarray(mask)
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    valid = (mask >= 0) & (mask < len(CLASS_COLORS))
    rgb[valid] = CLASS_COLORS[mask[valid].astype(np.int64)]
    rgb[~valid] = [0, 0, 0]
    return rgb


def _tile_starts(length, tile_size, stride):
    if length <= tile_size:
        return [0]
    starts = list(range(0, length - tile_size + 1, stride))
    last = length - tile_size
    if starts[-1] != last:
        starts.append(last)
    return starts


@torch.inference_mode()
def predict_array(image):
    """Segment an RGB image tile by tile, retaining its original output dimensions."""
    image = np.asarray(image.convert("RGB") if isinstance(image, Image.Image) else image)
    height, width = image.shape[:2]
    tile_size = IMAGE_SIZE
    overlap = min(64, tile_size // 4)
    stride = tile_size - overlap
    # Keep memory proportional to image size, rather than NUM_CLASSES x image size.
    best_score = np.full((height, width), -1.0, dtype=np.float32)
    prediction = np.zeros((height, width), dtype=np.uint8)
    model.eval()

    for top in _tile_starts(height, tile_size, stride):
        for left in _tile_starts(width, tile_size, stride):
            tile = image[top:min(top + tile_size, height), left:min(left + tile_size, width)]
            tile_h, tile_w = tile.shape[:2]
            pad_h, pad_w = tile_size - tile_h, tile_size - tile_w
            if pad_h or pad_w:
                tile = np.pad(tile, ((0, pad_h), (0, pad_w), (0, 0)), mode="edge")
            transformed = val_transform(image=tile, mask=np.zeros((tile_size, tile_size), dtype=np.uint8))
            pixels = transformed["image"].unsqueeze(0).to(DEVICE)
            logits = model(pixel_values=pixels).logits
            logits = torch.nn.functional.interpolate(logits, size=(tile_size, tile_size), mode="bilinear", align_corners=False)
            probs = torch.softmax(logits, dim=1)[0, :, :tile_h, :tile_w]
            tile_score, tile_prediction = probs.max(dim=0)
            tile_score = tile_score.cpu().numpy()
            tile_prediction = tile_prediction.cpu().numpy().astype(np.uint8)
            score_view = best_score[top:top + tile_h, left:left + tile_w]
            prediction_view = prediction[top:top + tile_h, left:left + tile_w]
            update = tile_score > score_view
            score_view[update] = tile_score[update]
            prediction_view[update] = tile_prediction[update]

    return image, prediction


def predict(image_path):
    return predict_array(Image.open(image_path).convert("RGB"))

# Quick validation example; the upload cell below lets you select your own image.
image_path, mask_path = val_pairs[0]
image, prediction = predict(image_path)
true_mask = np.array(Image.open(mask_path).convert("L"))
true_mask = np.array(Image.fromarray(true_mask.astype(np.uint8)).resize((image.shape[1], image.shape[0]), Image.Resampling.NEAREST))
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(image); axes[0].set_title("Input image")
axes[1].imshow(colorize_mask(true_mask)); axes[1].set_title("Ground truth")
axes[2].imshow(colorize_mask(prediction)); axes[2].set_title("SegFormer prediction")
for ax in axes: ax.axis("off")
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=CLASS_COLORS[i] / 255, label=name) for i, name in enumerate(CLASS_NAMES)]
fig.legend(handles=legend_handles, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout(); plt.show()


## 10. Upload and classify your own drone or satellite image

Run this after training has completed (or after loading a trained checkpoint). Colab will open a file picker. The code processes large images as overlapping tiles, keeps the original output dimensions, displays the segmentation, and downloads both a color mask and an overlay. Only categories represented in your labeled training masks can be learned. Results are only as reliable as the labeled training examples; use class 10 for areas truly hidden by smoke/fog/cloud, class 11 for industrial/factory sites, and class 12 for floodwater in the ground-truth masks.


In [ ]:
from google.colab import files
from IPython.display import display
import io

uploaded = files.upload()  # Select one or more RGB satellite/drone images
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for filename, file_bytes in uploaded.items():
    try:
        source_image = Image.open(io.BytesIO(file_bytes)).convert("RGB")
    except Exception as exc:
        print(f"Skipping {filename}: could not read image ({exc})")
        continue

    image, prediction = predict_array(source_image)
    colored_mask = colorize_mask(prediction)
    overlay = (0.55 * image.astype(np.float32) + 0.45 * colored_mask.astype(np.float32)).clip(0, 255).astype(np.uint8)
    stem = Path(filename).stem
    mask_path_out = OUTPUT_DIR / f"{stem}_segmentation.png"
    overlay_path_out = OUTPUT_DIR / f"{stem}_overlay.png"
    Image.fromarray(colored_mask).save(mask_path_out)
    Image.fromarray(overlay).save(overlay_path_out)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(image); axes[0].set_title("Uploaded image")
    axes[1].imshow(colored_mask); axes[1].set_title("Predicted classes")
    axes[2].imshow(overlay); axes[2].set_title("Overlay")
    for ax in axes: ax.axis("off")
    legend_handles = [plt.Rectangle((0, 0), 1, 1, color=CLASS_COLORS[i] / 255, label=name) for i, name in enumerate(CLASS_NAMES)]
    fig.legend(handles=legend_handles, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.04))
    plt.tight_layout(); plt.show()
    print(f"Saved full-resolution outputs: {mask_path_out} and {overlay_path_out}")
    files.download(str(mask_path_out))
    files.download(str(overlay_path_out))


## 11. Save and reuse the model

The best checkpoint is saved under `outputs/best_segformer`. To reload it in a later session, keep `CLASS_NAMES` consistent with the training run. For publication-quality reporting, evaluate on a held-out test set and include per-class IoU, dataset provenance, and the train/validation split method.

In [ ]:
# Reload after restarting the runtime:
# model = SegformerForSemanticSegmentation.from_pretrained("outputs/best_segformer").to(DEVICE)

# Optional: save the class names alongside the checkpoint for reproducibility.
(OUTPUT_DIR / "class_names.json").write_text(json.dumps(CLASS_NAMES, indent=2))
print("Checkpoint:", OUTPUT_DIR / "best_segformer")
